# Fine-tune Asisten PC — LoRA Qwen2.5-3B-Instruct

Latih model lokal agar **perilakunya** sesuai asisten kita: memilih tool yang tepat,
menolak tugas berat, dan menjawab percakapan ringan.

## Cara pakai di Kaggle
1. Upload `dataset.jsonl` sebagai Kaggle Dataset bernama **`asisten-pc-dataset`**.
2. Notebook ini: **Add Input** → pilih dataset tersebut.
3. Settings → **Accelerator: GPU T4 x2** (cukup satu T4, pilih T4 x2 jika tersedia).
4. Run All. Estimasi: 15–40 menit.
5. File hasil: `/kaggle/working/asisten-lora/` → download.

Tidak perlu kartu kredit. Kuota GPU Kaggle gratis ~30 jam/minggu.

In [ ]:
!pip -q install --retries 10 transformers peft accelerate || true

In [ ]:
import json, os, torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU T4 tidak terpasang. Cek pengaturan Accelerator di notebook Kaggle.")

In [ ]:
candidates = list(Path("/kaggle/input").glob("*/dataset.jsonl"))
candidates += list(Path("/kaggle/input").glob("**/dataset.jsonl"))
DATA_PATH = str(candidates[0]) if candidates else "/kaggle/working/dataset.jsonl"
print("Dataset:", DATA_PATH)

records = [json.loads(line) for line in Path(DATA_PATH).read_text(encoding="utf-8").splitlines() if line.strip()]
print("Jumlah contoh:", len(records))
print(records[0]["messages"][1]["content"])

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_LEN = 640

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class SFTData(Dataset):
    def __init__(self, rows, tok, max_len):
        self.examples = []
        for row in rows:
            messages = row["messages"]
            prompt = tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
            answer = messages[-1]["content"] + tok.eos_token
            prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
            answer_ids = tok(answer, add_special_tokens=False)["input_ids"]
            ids = (prompt_ids + answer_ids)[:max_len]
            labels = ([-100] * len(prompt_ids) + answer_ids)[:max_len]
            if len(ids) < 16:
                continue
            self.examples.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids),
                "labels": labels,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

train_data = SFTData(records, tokenizer, MAX_LEN)
print("Contoh ter-tokenisasi:", len(train_data))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
model.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir="/kaggle/working/asisten-out",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    optim="adamw_torch",
    max_grad_norm=0.3,
    group_by_length=True,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
trainer.train()

In [ ]:
OUT = "/kaggle/working/asisten-lora"
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print("Adapter tersimpan di", OUT)
for name in sorted(os.listdir(OUT)):
    print(" -", name)

## Uji cepat di notebook

Kalau keluarnya JSON seperti `{"tool": "get_specs", ...}`, berarti latihan berhasil.

In [ ]:
system = Path("/kaggle/working/PERSONA.md")
# System prompt bisa disalin dari file PERSONA.md + daftar tool di repo.
from transformers import pipeline

prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "Kamu adalah asisten PC. Balas HANYA JSON valid: {"tool": ..., "args": ..., "answer": ...}. Tool: get_specs(), list_dir(path), disk_usage(path), find_duplicates(path)."},
        {"role": "user", "content": "cek spec pc"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Memakai adapter di PC lokal

1. Download folder `asisten-lora` dari Kaggle.
2. Konversi adapter ke GGUF mengikuti dokumentasi llama.cpp
   (`convert_lora_to_gguf.py --base <model.gguf> --lora <adapter> --outfile adapter.gguf`).
3. Jalankan:
   ```
   llama-server.exe -m models\qwen2.5-3b-instruct-q4_k_m.gguf --lora adapter.gguf --port 8080 -c 4096
   ```
4. Buka asisten seperti biasa. Otak lokal sekarang sudah "terlatih" perilakunya.

Catatan: jika adapter GGUF tidak kompatibel, opsi alternatif adalah **merge** adapter ke
model dengan PEFT lalu konversi model hasil merge ke GGUF.